In [5]:
import os
os.environ['HF_HOME'] = './hf_cache'

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# 1. Dataset Loading (Path fixed for sample_data folder)
print("1. Loading dataset & preparing inputs...", flush=True)
file_path = "sample_data/merged_deduplicated_data.csv"
if not os.path.exists(file_path):
    file_path = "merged_deduplicated_data.csv"

df = pd.read_csv(file_path)

# Merging 'Market' and 'Business' into a single high-precision category
df['Category'] = df['Category'].replace({'Market': 'Business & Market', 'Business': 'Business & Market'})
df['Category'] = df['Category'].astype('category')
df['label'] = df['Category'].cat.codes
label_mapping = dict(enumerate(df['Category'].cat.categories))

# Construct richer input text (Title + Description/Content/URL)
content_col = 'Content' if 'Content' in df.columns else ('Description' if 'Description' in df.columns else 'URL')
df['text'] = df['Title'].fillna('').astype(str) + " " + df[content_col].fillna('').astype(str)

X = df['text'].to_numpy()
y = df['label'].to_numpy()

print("\n--- Updated Category Distribution ---")
print(df['Category'].value_counts())
print("--------------------------------------\n")

# 2. Stratified Data Split (80% Train, 10% Val, 10% Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# Compute Balanced Class Weights for Cross Entropy Loss
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# 3. RoBERTa Setup
MODEL_NAME = 'roberta-base'
MAX_LEN = 128
print(f"2. Loading {MODEL_NAME} with MAX_LEN={MAX_LEN}...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

train_loader = DataLoader(NewsDataset(X_train, y_train, tokenizer, MAX_LEN), batch_size=16, shuffle=True)
val_loader = DataLoader(NewsDataset(X_val, y_val, tokenizer, MAX_LEN), batch_size=32, shuffle=False)
test_loader = DataLoader(NewsDataset(X_test, y_test, tokenizer, MAX_LEN), batch_size=32, shuffle=False)

# 4. Model Setup & Device Allocation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Connected Device: {device}", flush=True)

num_classes = len(label_mapping)
class_weights = class_weights.to(device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    hidden_dropout_prob=0.2
)
model.to(device)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)

# 5. Optimizer & Cosine Warmup Scheduler
epochs = 5
optimizer = AdamW(model.parameters(), lr=1.5e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

# 6. GPU Accelerated Training Loop
print("\n--- Training Started ---", flush=True)

for epoch in range(epochs):
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    # Validation Step
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    val_acc = accuracy_score(val_targets, val_preds)
    print(f">> Epoch {epoch+1}/{epochs} Validation Accuracy: {val_acc*100:.2f}%", flush=True)

# 7. Final Model Evaluation
print("\n==========================================", flush=True)
print("     FINAL RoBERTa EVALUATION REPORT      ", flush=True)
print("==========================================", flush=True)

model.eval()
test_preds, test_targets = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(labels.cpu().numpy())

overall_acc = accuracy_score(test_targets, test_preds)
print(f"\n>>> OVERALL TEST ACCURACY: {overall_acc*100:.2f}%\n", flush=True)

target_names = [label_mapping[i] for i in range(num_classes)]
print("--- Detailed Classification Report ---", flush=True)
print(classification_report(test_targets, test_preds, target_names=target_names), flush=True)

1. Loading dataset & preparing inputs...

--- Updated Category Distribution ---
Category
Business & Market    3131
Politics              769
Technology            390
Health                150
Energy                136
Name: count, dtype: int64
--------------------------------------

2. Loading roberta-base with MAX_LEN=128...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Training Started ---
>> Epoch 1/5 Validation Accuracy: 79.48%
>> Epoch 2/5 Validation Accuracy: 79.91%
>> Epoch 3/5 Validation Accuracy: 85.81%
>> Epoch 4/5 Validation Accuracy: 84.28%
>> Epoch 5/5 Validation Accuracy: 84.06%

     FINAL RoBERTa EVALUATION REPORT      

>>> OVERALL TEST ACCURACY: 86.90%

--- Detailed Classification Report ---
                   precision    recall  f1-score   support

Business & Market       0.96      0.88      0.92       314
           Energy       0.61      0.85      0.71        13
           Health       0.85      0.73      0.79        15
         Politics       0.80      0.86      0.83        77
       Technology       0.59      0.87      0.70        39

         accuracy                           0.87       458
        macro avg       0.76      0.84      0.79       458
     weighted avg       0.89      0.87      0.87       458

